# Lesson 2: NumPy

One operation on a whole column of numbers.

<img src="../img/map-2.svg" width="85%" alt="">

## The cost of a loop

Project 1 read 50,000 events, and you waited a moment. The full sample here is 300,000 events, and an LHC experiment records billions.

Every analysis asks the same question of every event, so the loop is always the same shape. Array programming hands that loop to fast compiled code: you write the operation once, the library applies it to the whole column.

NumPy is the shared language of scientific Python: pandas, hist, SciPy and Awkward Array all build on its arrays. Learn the moves once, use them everywhere.

## Loops and arrays

A loop handles one value at a time. An array handles the whole column.

<img src="../img/loop-vs-column.svg" width="85%" alt="">

## This style has a name

Spelling out the loop is *imperative* programming. Stating one operation on whole columns, with no index and no element in sight, is *array-oriented* programming.

Every array language, from APL in the 1960s to MATLAB, R and NumPy, was built for the same job: interactive data analysis. Short questions, quick answers, at any data size.

They are styles, not laws. Python lets you mix them, and this lesson is about when the array style wins.

## Arrays

An array is one block of values that all have the same type.

Everyone imports NumPy under the short name `np`.

<img src="../img/list-vs-array.svg" width="80%" alt="">

In [ ]:
import numpy as np

pt = np.array([63.0, 38.1, 4.0, 54.3])
pt.dtype, pt.shape

## Making arrays

From a list, from a range, from nothing.

In [ ]:
np.arange(5), np.linspace(0.0, 1.0, 5)

In [ ]:
np.zeros(4), np.ones(3)

## Every array has a `dtype`

One type for the whole block, and the operations follow it. `astype` converts the whole column at once.

In [ ]:
counts = np.array([5, 2, 9])
counts.dtype, (counts / 2).dtype

In [ ]:
mask = np.array([True, False, True])
mask.astype(np.int32)

## Why arrays are fast

A list stores pointers to number objects scattered through memory. An array stores the raw numbers side by side, in one block.

One operation on an array is one call into compiled code, which sweeps the block without touching Python again until it is done. That is the speed you will measure in a moment.

<img src="../img/memory-layout.svg" width="85%" alt="">


Memory is the dtype times the count, nothing more. `astype` trades precision for half the space:

In [ ]:
big = np.ones(1_000_000)
big.nbytes, big.astype(np.float32).nbytes

## Puzzle: who shares the memory?

Guess before you run: after the last line, what is in `a`?

<img src="../img/view.svg" width="70%" alt="">

In [ ]:
a = np.array([63.0, 38.1, 4.0])
b = a[:2]
b[0] = 999.0
a

A slice of an array is a *view*: the same memory under a new name. Fast, because nothing is copied — and a surprise, until you know it. `a[:2].copy()` gives you your own block. (A slice of a *list* is a copy. Python keeps you alert.)

## Arithmetic

Arithmetic applies to every element at once.

<img src="../img/elementwise.svg" width="70%" alt="">

In [ ]:
pt * 2

## Broadcasting

Shapes do not have to match: a single number is reused for every element, and a row and a column spread out into a table.

<img src="../img/broadcast.svg" width="80%" alt="">


In [ ]:
pt - 20.0

In [ ]:
pt2d = np.array([[63.0, 38.1, 4.0],
                 [54.3, 23.5, 52.9]])
scale = np.array([1.0, 1.0, 0.5])

pt2d * scale

## Lists are different

A list does something else with `* 2`: it repeats itself.

In [ ]:
[63.0, 38.1, 4.0, 54.3] * 2

## NumPy functions

NumPy functions apply to every element. Arrays of the same shape combine element by element.

The momentum along the beam is $p_z = p_T \sinh\eta$.

<img src="../img/elementwise-two.svg" width="70%" alt="">

In [ ]:
eta = np.array([-0.72, -0.88, -0.32, -1.06])
pz = pt * np.sinh(eta)
pz

## Puzzle: the mass that is not there

In project 1, `mass()` took a square root. What if the number under it is negative?

Guess before you run: an error, or something else?

<img src="../img/nan-flow.svg" width="85%" alt="">


In [ ]:
np.sqrt(np.array([4.0, -1.0]))

`nan` — *not a number* — with a warning, not an error. It flows silently through every later step, and `nan == nan` is `False`. When a mass distribution has a hole, look for a `nan` upstream.

## Speed

The array version is much faster than the loop.

`%timeit` runs a line many times and reports the average time.

In [ ]:
big = np.arange(1_000_000)
%timeit [x * 2 for x in big]

In [ ]:
%timeit big * 2

## Masks

A comparison on an array gives a mask: one `True` or `False` per element.

In [ ]:
mask = pt > 20
mask

## Selecting with a mask

A mask inside square brackets keeps the `True` entries.

<img src="../img/mask.svg" width="70%" alt="">

In [ ]:
pt[mask]

## Indexing with positions

Square brackets also accept a list of positions: any order, repeats allowed. A mask asks "which pass?", positions say "these, in this order".

In [ ]:
pt[np.array([2, 0, 0])]

## Masks across arrays

A mask made from one array can select from another.

Position 0 in `pt` and position 0 in `eta` describe the same muon.

<img src="../img/mask-other.svg" width="70%" alt="">

In [ ]:
eta[mask]

## Combining masks

Combine masks with `&` (and) and `|` (or). Put each condition in parentheses.

In [ ]:
(pt > 20) & (np.abs(eta) < 1.0)

## What does not work

The words `and` and `or` do not work on arrays, and neither does `12 < x < 120`.

Read the last line of the error.

In [ ]:
(pt > 20) and (np.abs(eta) < 1.0)

## Counting

Counting the `True` values counts what passes.

`True` counts as 1 and `False` as 0. The mean of a mask is the fraction that passes.

In [ ]:
print(mask.sum(), mask.mean())

## Reductions

A reduction turns many values into one.

<img src="../img/reduce.svg" width="75%" alt="">

In [ ]:
print(pt.max(), pt.min(), len(pt))

## Which one, not how big

`max` says how big. `argmax` says *where*. The position then looks up anything else about the winner.

<img src="../img/argmax.svg" width="80%" alt="">


In [ ]:
i = np.argmax(pt)
i, eta[i]

## Puzzle: the length of a curve

`x` and `y` sample a parabola. How long is the curve they trace?

No loop: `x[1:] - x[:-1]` is every step, all at once. Pythagoras on the steps, then one sum. Try it in the workbook; the answer is close to 4.6468.

<img src="../img/curve-length.svg" width="80%" alt="">


In [ ]:
x = np.linspace(0, 2, 1000)
y = x**2

np.sum(np.sqrt((x[1:] - x[:-1])**2 + (y[1:] - y[:-1])**2))

## `np.minimum` and `np.maximum`

`np.minimum` and `np.maximum` compare two arrays element by element.

In [ ]:
other = np.array([50.0, 20.0, 10.0, 60.0])
np.minimum(pt, other)

## Two dimensions

An array can have two dimensions. `axis` names the dimension that a reduction removes.

Here each row is an event. `axis=1` leaves one value per event.

<img src="../img/axis.svg" width="70%" alt="">

In [ ]:
pt2d = np.array([[63.0, 38.1, 4.0],
                 [54.3, 23.5, 52.9]])
pt2d.sum(axis=1)

`[row, column]` picks one value, and `:` means the whole axis. `pt2d[:, 0]` is the first column: one value per row.

In [ ]:
pt2d[0, 2], pt2d[:, 0]

## Puzzle: hit the target

`m` is 5 × 6. Find the single slice `m[?, ?]` that gives

```python
[[ 4,  5],
 [28, 29]]
```

In [ ]:
m = np.arange(30).reshape(5, 6)
m

In [ ]:
m[::4, 4:]

## Invent data to test with

A generator with a seed gives the same "random" numbers on every run, so a test you write today fails the same way tomorrow.

In [ ]:
rng = np.random.default_rng(1)
rng.normal(91.2, 2.5, 5)

## Histograms

The `hist` library builds, fills and plots a histogram.

`Reg(50, 0, 100)` means 50 bins of equal width from 0 to 100.

<img src="../img/histogram.svg" width="65%" alt="">

In [ ]:
import hist

h = hist.Hist.new.Reg(50, 0, 100, label="pT [GeV]").Double()
h.fill(np.random.default_rng(1).exponential(20, 10_000))
h.plot();

## Puzzle: guess the shape

One die rolls flat: every face equally often. Before you run it: what does the *sum of two* dice look like — still flat?

In [ ]:
rolls = rng.integers(1, 7, 100_000) + rng.integers(1, 7, 100_000)
hist.Hist.new.Reg(11, 1.5, 12.5, label="sum of two dice").Double().fill(rolls).plot();

Sums pile up in the middle: there is one way to roll 2, six ways to roll 7. Add more dice and the triangle becomes the bell curve. Most measurement noise is a sum of many small effects, which is why the bell curve is everywhere.

## What you can now read

| You see | It means |
|---|---|
| `np.array([...])` | make an array |
| `pt * 2`, `np.sqrt(pt)` | apply to every element |
| `pt > 20` | a mask of `True` and `False` |
| `pt[mask]` | keep the `True` entries |
| `(a > 1) & (b < 2)` | both conditions |
| `mask.sum()` | how many pass |
| `np.minimum(a, b)` | the smaller of two arrays, element by element |
| `pt - 20.0` | the number is reused for every element |
| `pt[np.array([2, 0])]` | pick by position |
| `np.arange(5)`, `np.zeros(4)` | make an array from nothing |
| `x.astype(np.int32)` | convert the whole column |
| `pt2d[:, 0]` | the first column |
| `np.argmax(pt)` | the position of the biggest |


## Now: project 2.

[project-higgs.ipynb](project-higgs.ipynb): the same analysis on whole columns.

<img src="../img/lepton-columns.svg" width="65%" alt="">